# S03 — Model Evaluation (Quantile LightGBM)

Visual QA for the Phase 3 asymmetric quantile model (α=0.90).  
The model intentionally predicts the **90th percentile** of risk — pessimistic by design because missing a blackout (FN) costs 10× more than a false alarm (FP).

In [ ]:
import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

from grid_risk.features import FEATURE_NAMES, RiskIndexFit
from grid_risk.model import (
    assign_categories,
    baseline_relative_rmse,
    cost_matrix_penalty,
    evaluate,
    f_beta_high,
    get_feature_importance,
    peak_window_accuracy,
    persistence_baseline,
    predict,
)

DATA = ROOT / "data"
ARTIFACTS = DATA / "artifacts"

In [ ]:
# Load data and models
test = pd.read_parquet(DATA / "test.parquet")
fit: RiskIndexFit = joblib.load(ARTIFACTS / "risk_index_fit.joblib")
lgbm = joblib.load(ARTIFACTS / "lgbm_model.joblib")
ridge = joblib.load(ARTIFACTS / "ridge_model.joblib")
metrics = json.loads((ARTIFACTS / "metrics.json").read_text())
feat_imp = pd.read_csv(ARTIFACTS / "feature_importance.csv")

thresholds = fit.thresholds
X_test = test[FEATURE_NAMES]
y_test = test["risk_index"].values
y_pred = predict(lgbm, X_test)
y_baseline = test["risk_index_lag_24h"].values

test["predicted"] = y_pred
test["true_cat"] = assign_categories(y_test, thresholds)
test["pred_cat"] = assign_categories(y_pred, thresholds)
test["baseline_cat"] = assign_categories(y_baseline, thresholds)
test["residual"] = y_pred - y_test

print(f"Test set: {len(test)} hours ({test.index[0].date()} to {test.index[-1].date()})")
print(f"Thresholds: Low ≤ {thresholds[0]:.3f} < Medium ≤ {thresholds[1]:.3f} < High")

## 1. Metrics Summary Table

In [ ]:
tm = metrics["test"]
summary = pd.DataFrame([
    {"Metric": "A. Peak Window Accuracy (±1h)", "Value": f"{tm['peak_window_accuracy']*100:.1f}%"},
    {"Metric": "B. Cost Matrix Penalty", "Value": f"{tm['cost_penalty']} pts (baseline: {tm['baseline_cost_penalty']}, reduction: {tm['cost_reduction_pct']:.1f}%)"},
    {"Metric": "C. F₃ Score (High class)", "Value": f"{tm['f3_high']:.3f}"},
    {"Metric": "D. RMSE Skill Score", "Value": f"{tm['rmse_skill_score']:.1f}% vs persistence"},
    {"Metric": "RMSE", "Value": f"{tm['rmse']:.4f}"},
    {"Metric": "MAE", "Value": f"{tm['mae']:.4f}"},
    {"Metric": "R²", "Value": f"{tm['r2']:.4f}"},
])
summary.style.hide(axis="index")

## 2. Actual vs Predicted Scatter

In [ ]:
cat_order = ["Low", "Medium", "High"]
color_map = {"Low": "#2ecc71", "Medium": "#f39c12", "High": "#e74c3c"}

fig = px.scatter(
    test, x="risk_index", y="predicted", color="true_cat",
    category_orders={"true_cat": cat_order},
    color_discrete_map=color_map,
    opacity=0.3, labels={"risk_index": "Actual", "predicted": "Predicted", "true_cat": "Actual Category"},
    title="Actual vs Predicted Risk Index (quantile α=0.90)",
)
fig.add_shape(type="line", x0=0, x1=1, y0=0, y1=1, line=dict(dash="dash", color="gray"))
fig.update_layout(width=700, height=600)
fig.show()

## 3. 1-Week Time Series Overlay

In [ ]:
# Pick a representative week from the test set
week_start = test.index[len(test) // 4]
week_end = week_start + pd.Timedelta(days=7)
week = test.loc[week_start:week_end]

fig = go.Figure()
fig.add_trace(go.Scatter(x=week.index, y=week["risk_index"], name="Actual", line=dict(color="#2c3e50")))
fig.add_trace(go.Scatter(x=week.index, y=week["predicted"], name="Predicted (Q90)", line=dict(color="#e74c3c", dash="dash")))
fig.add_trace(go.Scatter(x=week.index, y=week["risk_index_lag_24h"], name="Persistence", line=dict(color="#95a5a6", dash="dot")))

# Add threshold bands
fig.add_hline(y=thresholds[0], line_dash="dot", line_color="green", annotation_text="Low/Med")
fig.add_hline(y=thresholds[1], line_dash="dot", line_color="orange", annotation_text="Med/High")

fig.update_layout(
    title=f"1-Week Overlay: {week_start.date()} to {week_end.date()}",
    yaxis_title="Risk Index", xaxis_title="Time (UTC)",
    width=900, height=450,
)
fig.show()

## 4. Residual Distribution & Residuals vs Predicted

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["Residual Distribution", "Residuals vs Predicted"])

fig.add_trace(
    go.Histogram(x=test["residual"], nbinsx=80, marker_color="#3498db", name="Residuals"),
    row=1, col=1,
)
fig.add_vline(x=0, line_dash="dash", line_color="red", row=1, col=1)
fig.add_vline(x=test["residual"].median(), line_dash="dot", line_color="orange", row=1, col=1)

fig.add_trace(
    go.Scatter(x=test["predicted"], y=test["residual"], mode="markers",
               marker=dict(color="#3498db", opacity=0.2, size=3), name="Residuals"),
    row=1, col=2,
)
fig.add_hline(y=0, line_dash="dash", line_color="red", row=1, col=2)

fig.update_layout(width=1000, height=400, showlegend=False,
                  title="Residuals (predicted − actual): positive bias expected for Q90")
fig.update_xaxes(title_text="Residual", row=1, col=1)
fig.update_xaxes(title_text="Predicted", row=1, col=2)
fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_yaxes(title_text="Residual", row=1, col=2)
fig.show()

print(f"Median residual: {test['residual'].median():.4f} (positive = pessimistic bias, as designed)")
print(f"% positive residuals: {(test['residual'] > 0).mean()*100:.1f}%")

## 5. Confusion Matrix (3×3)

In [ ]:
from sklearn.metrics import confusion_matrix

labels = ["Low", "Medium", "High"]
cm = confusion_matrix(test["true_cat"], test["pred_cat"], labels=labels)

fig = px.imshow(
    cm, x=labels, y=labels, text_auto=True,
    color_continuous_scale="YlOrRd",
    labels=dict(x="Predicted", y="Actual", color="Count"),
    title="Confusion Matrix — Quantile LightGBM (α=0.90)",
)
fig.update_layout(width=500, height=500)
fig.show()

# Key insight: FN for High class
high_fn = cm[2, 0] + cm[2, 1]
high_total = cm[2, :].sum()
print(f"\nHigh class recall: {cm[2,2]/high_total*100:.1f}% ({high_fn} FN out of {high_total} actual High hours)")

## 6. Peak Window Accuracy — Daily Hits/Misses

In [ ]:
df_peak = test[["risk_index", "predicted"]].copy()
df_peak["date"] = df_peak.index.tz_convert("Europe/Madrid").date
df_peak["madrid_hour"] = df_peak.index.tz_convert("Europe/Madrid").hour

daily_peaks = []
for date, day_df in df_peak.groupby("date"):
    if len(day_df) < 6:
        continue
    actual_hour = day_df.loc[day_df["risk_index"].idxmax(), "madrid_hour"]
    pred_hour = day_df.loc[day_df["predicted"].idxmax(), "madrid_hour"]
    hit = abs(pred_hour - actual_hour) <= 1
    daily_peaks.append({"date": date, "actual_peak_h": actual_hour, "pred_peak_h": pred_hour, "hit": hit})

peaks_df = pd.DataFrame(daily_peaks)
peaks_df["date"] = pd.to_datetime(peaks_df["date"])

fig = px.scatter(
    peaks_df, x="date", y="actual_peak_h", color="hit",
    color_discrete_map={True: "#2ecc71", False: "#e74c3c"},
    labels={"date": "Date", "actual_peak_h": "Actual Peak Hour", "hit": "Within ±1h"},
    title=f"Peak Window Accuracy: {peaks_df['hit'].mean()*100:.1f}% of days within ±1h",
)
fig.update_layout(width=900, height=400)
fig.show()

## 7. Cost Matrix Breakdown

In [ ]:
# Model costs
fn_model = sum(1 for a, p in zip(test["true_cat"], test["pred_cat"]) if a == "High" and p != "High")
fp_model = sum(1 for a, p in zip(test["true_cat"], test["pred_cat"]) if a != "High" and p == "High")

# Baseline costs
fn_base = sum(1 for a, p in zip(test["true_cat"], test["baseline_cat"]) if a == "High" and p != "High")
fp_base = sum(1 for a, p in zip(test["true_cat"], test["baseline_cat"]) if a != "High" and p == "High")

cost_data = pd.DataFrame({
    "Type": ["FN (×10)", "FP (×1)", "Total"],
    "Model": [fn_model * 10, fp_model * 1, fn_model * 10 + fp_model],
    "Persistence": [fn_base * 10, fp_base * 1, fn_base * 10 + fp_base],
})

fig = go.Figure()
fig.add_trace(go.Bar(name="Model", x=["FN penalty (×10)", "FP penalty (×1)"], y=[fn_model*10, fp_model], marker_color="#e74c3c"))
fig.add_trace(go.Bar(name="Persistence", x=["FN penalty (×10)", "FP penalty (×1)"], y=[fn_base*10, fp_base], marker_color="#95a5a6"))
fig.update_layout(barmode="group", title="Cost Matrix Breakdown: Model vs Persistence", width=600, height=400,
                  yaxis_title="Penalty Points")
fig.show()

print(cost_data.to_string(index=False))
print(f"\nModel: {fn_model} FN hours, {fp_model} FP hours")
print(f"Persistence: {fn_base} FN hours, {fp_base} FP hours")

## 8. Feature Importance

In [ ]:
fig = px.bar(
    feat_imp, x="importance", y="feature", orientation="h",
    title="LightGBM Feature Importance (split-based)",
    labels={"importance": "Importance", "feature": "Feature"},
    color="importance", color_continuous_scale="YlOrRd",
)
fig.update_layout(width=700, height=400, yaxis=dict(autorange="reversed"), showlegend=False)
fig.show()

## 9. SHAP Summary Plot

In [ ]:
try:
    import shap

    explainer = shap.TreeExplainer(lgbm)
    shap_values = explainer.shap_values(X_test.iloc[:2000])  # subsample for speed

    fig, ax = plt.subplots(figsize=(10, 5))
    shap.summary_plot(shap_values, X_test.iloc[:2000], show=False)
    plt.title("SHAP Summary — Top Feature Contributions")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("shap not installed — run: uv sync --extra notebooks")

## 10. Baseline Comparison

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

mask = ~np.isnan(y_baseline)
y_ridge = np.clip(ridge.predict(X_test), 0, 1)

comparison = pd.DataFrame({
    "Model": ["Persistence (24h lag)", "Ridge", "LightGBM (Q90)"],
    "RMSE": [
        np.sqrt(mean_squared_error(y_test[mask], y_baseline[mask])),
        np.sqrt(mean_squared_error(y_test, y_ridge)),
        np.sqrt(mean_squared_error(y_test, y_pred)),
    ],
    "MAE": [
        mean_absolute_error(y_test[mask], y_baseline[mask]),
        mean_absolute_error(y_test, y_ridge),
        mean_absolute_error(y_test, y_pred),
    ],
    "Cost Penalty": [
        cost_matrix_penalty(assign_categories(y_test[mask], thresholds), assign_categories(y_baseline[mask], thresholds)),
        cost_matrix_penalty(assign_categories(y_test, thresholds), assign_categories(y_ridge, thresholds)),
        cost_matrix_penalty(assign_categories(y_test, thresholds), assign_categories(y_pred, thresholds)),
    ],
    "F₃ (High)": [
        f_beta_high(assign_categories(y_test[mask], thresholds), assign_categories(y_baseline[mask], thresholds)),
        f_beta_high(assign_categories(y_test, thresholds), assign_categories(y_ridge, thresholds)),
        f_beta_high(assign_categories(y_test, thresholds), assign_categories(y_pred, thresholds)),
    ],
})

comparison.style.format({
    "RMSE": "{:.4f}", "MAE": "{:.4f}", "Cost Penalty": "{:,}", "F₃ (High)": "{:.3f}"
}).highlight_min(subset=["RMSE", "MAE", "Cost Penalty"], color="#d5f5e3"
).highlight_max(subset=["F₃ (High)"], color="#d5f5e3")